In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Source (Bronze) and Target (Silver)
bronze_table = "workspace.default.raw_customer"
silver_table = "workspace.default.silver_customer"

# Read Bronze Data
df = spark.table(bronze_table)

# Data Cleansing & Standardization
silver_df = (
    df
    .withColumn("customerID", F.col("customerID").cast("long"))
    .withColumn("first_name", F.initcap(F.trim(F.col("first_name"))))
    .withColumn("last_name", F.initcap(F.trim(F.col("last_name"))))
    .withColumn("email_address", F.lower(F.trim(F.col("email_address"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("state", F.upper(F.trim(F.col("state"))))
    .withColumn("country", F.initcap(F.trim(F.col("country"))))
    .withColumn("continent", F.initcap(F.trim(F.col("continent"))))
    .withColumn("gender", F.lower(F.trim(F.col("gender"))))
)

# Remove Duplicate Customer IDs
window_spec = Window.partitionBy("customerID").orderBy(
    F.col("bronze_ingestion_ts").desc()
)

silver_df = (
    silver_df
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Remove Records with Null Customer ID
silver_df = silver_df.filter(F.col("customerID").isNotNull())

silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)

# Add Silver Metadata
